In [3]:
import os
import math
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Change to project root
if Path.cwd().name == "lstm":
    os.chdir("../..")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

# Repo paths
REPO_ROOT = Path.home() / "pv_forecast_30d"
DATA_DIR  = REPO_ROOT / "data" / "processed" / "pretraining" / "germany" / "global"

TRAIN_PQ  = DATA_DIR / "regional_train.parquet"
VAL_PQ    = DATA_DIR / "regional_val.parquet"
SCALER_JS = DATA_DIR / "regional_scaler.json"

ENCODER_PT = REPO_ROOT / "experiments" / "lstm" / "encoders" / "lstm_encoder_germany_regional_CANONICAL.pt"

print("TRAIN:", TRAIN_PQ, TRAIN_PQ.exists())
print("VAL:  ", VAL_PQ, VAL_PQ.exists())
print("SCALER:", SCALER_JS, SCALER_JS.exists())
print("ENCODER:", ENCODER_PT, ENCODER_PT.exists())

# Import schema + model
from src.data.schema import (
    TIME_COL, PLANT_ID_COL, TARGET_COL,
    LSTM_INPUT_FEATURES, GLOBAL_LSTM_INPUT_FEATURES, PLANT_ONEHOT_COLS,
    TIME_STEP_MINUTES,
)
from src.models.global_lstm_encoder import GlobalLSTMEncoder, LSTMEncoderConfig

print("len(LSTM_INPUT_FEATURES):", len(LSTM_INPUT_FEATURES))
print("len(PLANT_ONEHOT_COLS):", len(PLANT_ONEHOT_COLS))
print("len(GLOBAL_LSTM_INPUT_FEATURES):", len(GLOBAL_LSTM_INPUT_FEATURES))


TRAIN: /home/dwijenayake/pv_forecast_30d/data/processed/pretraining/germany/global/regional_train.parquet True
VAL:   /home/dwijenayake/pv_forecast_30d/data/processed/pretraining/germany/global/regional_val.parquet True
SCALER: /home/dwijenayake/pv_forecast_30d/data/processed/pretraining/germany/global/regional_scaler.json True
ENCODER: /home/dwijenayake/pv_forecast_30d/experiments/lstm/encoders/lstm_encoder_germany_regional_CANONICAL.pt True
len(LSTM_INPUT_FEATURES): 15
len(PLANT_ONEHOT_COLS): 5
len(GLOBAL_LSTM_INPUT_FEATURES): 20


# Scaler loader + inverse z-score + RAW GTI helper

In [5]:
GTI_COL = "global_tilted_irradiance_instant"

def _looks_like_format_A(d: dict) -> bool:
    # {col: {"mean": m, "std": s}, ...}
    if not isinstance(d, dict) or len(d) == 0:
        return False
    v = next(iter(d.values()))
    return isinstance(v, dict) and ("mean" in v) and ("std" in v)

def _looks_like_format_B(d: dict) -> bool:
    # {"mean": {col: m}, "std": {col: s}}
    return (
        isinstance(d, dict)
        and "mean" in d and "std" in d
        and isinstance(d["mean"], dict) and isinstance(d["std"], dict)
        and len(d["mean"]) > 0
    )

def _looks_like_format_C(d: dict) -> bool:
    # {"columns":[...], "mean":[...], "std":[...]}
    return (
        isinstance(d, dict)
        and "columns" in d and "mean" in d and "std" in d
        and isinstance(d["columns"], list)
        and isinstance(d["mean"], list)
        and isinstance(d["std"], list)
        and len(d["columns"]) == len(d["mean"]) == len(d["std"])
        and len(d["columns"]) > 0
    )

def _normalize_to_col_stats(d: dict) -> dict:
    # Convert any of A/B/C into {col: {"mean": float, "std": float}}
    if _looks_like_format_A(d):
        out = {}
        for c, v in d.items():
            out[str(c)] = {"mean": float(v["mean"]), "std": float(v["std"])}
        return out

    if _looks_like_format_B(d):
        out = {}
        for c in d["mean"].keys():
            if c in d["std"]:
                out[str(c)] = {"mean": float(d["mean"][c]), "std": float(d["std"][c])}
        return out

    if _looks_like_format_C(d):
        out = {}
        for c, m, s in zip(d["columns"], d["mean"], d["std"]):
            out[str(c)] = {"mean": float(m), "std": float(s)}
        return out

    raise ValueError("Object does not match scaler formats A/B/C")

def load_scaler_stats(path):
    """
    Load scaler stats from JSON, even if nested.

    It searches recursively for an object matching one of:
      A) {col: {"mean": m, "std": s}, ...}
      B) {"mean": {col: m, ...}, "std": {col: s, ...}}
      C) {"columns":[...], "mean":[...], "std":[...]}
    """
    with open(path, "r") as f:
        obj = json.load(f)

    # quick debug peek (safe, not huge)
    if isinstance(obj, dict):
        print("regional_scaler.json top-level keys:", list(obj.keys())[:30])
    else:
        print("regional_scaler.json top-level type:", type(obj))

    # DFS search
    stack = [obj]
    visited = 0
    while stack:
        cur = stack.pop()
        visited += 1

        if isinstance(cur, dict):
            if _looks_like_format_A(cur) or _looks_like_format_B(cur) or _looks_like_format_C(cur):
                out = _normalize_to_col_stats(cur)
                print(f"[OK] Found scaler stats after visiting {visited} nodes. cols={len(out)}")
                return out

            # push children
            for v in cur.values():
                if isinstance(v, (dict, list)):
                    stack.append(v)

        elif isinstance(cur, list):
            for v in cur:
                if isinstance(v, (dict, list)):
                    stack.append(v)

    raise ValueError(f"Could not find scaler stats inside JSON: {path}")

def inverse_zscore(z: np.ndarray, mean: float, std: float) -> np.ndarray:
    return z * std + mean

stats = load_scaler_stats(SCALER_JS)
print("Has GTI key:", GTI_COL in stats)
if GTI_COL in stats:
    print("GTI mean/std:", stats[GTI_COL]["mean"], stats[GTI_COL]["std"])
else:
    # show similar keys to find naming mismatch
    near = [k for k in stats.keys() if "tilt" in k.lower() or "irradi" in k.lower() or "gti" in k.lower()]
    print("Closest irradiance-like keys:", near[:30])


regional_scaler.json top-level keys: ['normalized_columns', 'stats']
[OK] Found scaler stats after visiting 2 nodes. cols=14
Has GTI key: True
GTI mean/std: 112.04923558301667 166.8605708056494


# Cell 3 (window dataset for evaluation, returns X, y, pid_idx, t_int, gti_raw_at_target)

In [6]:
from torch.utils.data import Dataset, DataLoader

class EvalGroupedWindowDataset(Dataset):
    """
    Builds windows per plant, never crosses plants.
    Also returns:
      - pid_idx (int)
      - t_int (int64 ns) for the target timestamp
      - gti_raw_target (float32) reconstructed from z-scored GTI using regional_scaler.json
    """
    def __init__(self, df: pd.DataFrame, scaler_stats: dict, window_size: int = 96, stride: int = 1):
        self.window_size = int(window_size)
        self.stride = int(stride)

        d = df.copy()
        d[TIME_COL] = pd.to_datetime(d[TIME_COL], utc=True)
        d = d.sort_values([PLANT_ID_COL, TIME_COL]).reset_index(drop=True)

        required = set([TIME_COL, PLANT_ID_COL, TARGET_COL] + GLOBAL_LSTM_INPUT_FEATURES)
        missing = sorted(required - set(d.columns))
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        # map plant ids to ints
        plant_ids = sorted(d[PLANT_ID_COL].unique().tolist())
        self.pid2i = {pid: i for i, pid in enumerate(plant_ids)}
        self.i2pid = {i: pid for pid, i in self.pid2i.items()}

        # scaler GTI
        if GTI_COL not in scaler_stats:
            raise KeyError(f"Scaler stats missing '{GTI_COL}'. Check regional_scaler.json keys.")
        gti_mean = float(scaler_stats[GTI_COL]["mean"])
        gti_std = float(scaler_stats[GTI_COL]["std"])

        freq_s = int(TIME_STEP_MINUTES * 60)

        self._by_plant = {}
        self._index = []  # (pid_idx, start_idx)

        for pid, g in d.groupby(PLANT_ID_COL, sort=True):
            g = g.sort_values(TIME_COL).reset_index(drop=True)
            n = len(g)
            if n <= self.window_size:
                continue

            times_ns = g[TIME_COL].astype("int64").to_numpy()  # ns
            times_s = (times_ns // 10**9).astype(np.int64)

            X = g[GLOBAL_LSTM_INPUT_FEATURES].to_numpy(dtype=np.float32)
            y = g[TARGET_COL].to_numpy(dtype=np.float32)

            # reconstruct raw gti at each row from z-scored stored GTI
            gti_z = g[GTI_COL].to_numpy(dtype=np.float32)
            gti_raw = inverse_zscore(gti_z, gti_mean, gti_std).astype(np.float32)
            gti_raw = np.clip(gti_raw, 0.0, None)  # physical sanity, avoid negative after inverse scaling

            # regularity
            diffs = np.diff(times_s)
            good_step = (diffs == freq_s)

            max_start = n - self.window_size - 1
            pid_idx = self.pid2i[pid]

            self._by_plant[pid_idx] = {
                "X": X,
                "y": y,
                "times_ns": times_ns,
                "gti_raw": gti_raw,
            }

            for i in range(0, max_start + 1, self.stride):
                if good_step[i : i + self.window_size].all():
                    self._index.append((pid_idx, i))

        if len(self._index) == 0:
            raise ValueError("No valid windows created. Check window_size, gaps, row counts.")

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx: int):
        pid_idx, i = self._index[idx]
        pack = self._by_plant[pid_idx]

        X = pack["X"][i : i + self.window_size]
        y = pack["y"][i + self.window_size]
        t_ns = pack["times_ns"][i + self.window_size]
        gti_raw_target = pack["gti_raw"][i + self.window_size]

        return (
            torch.from_numpy(X),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(pid_idx, dtype=torch.long),
            torch.tensor(t_ns, dtype=torch.long),
            torch.tensor(gti_raw_target, dtype=torch.float32),
        )


# Cell 4 (load model, evaluation loop, persistence baseline, active-only)

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_encoder(weights_path: Path) -> torch.nn.Module:
    cfg = LSTMEncoderConfig(
        input_size=len(GLOBAL_LSTM_INPUT_FEATURES),
        hidden_size=64,
        num_layers=2,
        dropout=0.1,
        lr=1e-4,
    )
    m = GlobalLSTMEncoder(cfg)
    sd = torch.load(weights_path, map_location="cpu")
    m.load_state_dict(sd, strict=False)
    m.eval()
    return m.to(DEVICE)

def rmse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))

def eval_on_val(val_df: pd.DataFrame, model: torch.nn.Module, day_thr_raw: float = 10.0, power_thr: float = 0.02,
                window_size: int = 96, batch_size: int = 512, num_workers: int = 2) -> dict:
    ds = EvalGroupedWindowDataset(val_df, stats, window_size=window_size, stride=1)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    y_all = []
    yhat_all = []
    yhat_persist_all = []
    gti_raw_all = []

    with torch.no_grad():
        for X, y, pid_idx, t_ns, gti_raw in loader:
            X = X.to(DEVICE)

            out = model(X)
            if isinstance(out, dict):
                pred = out.get("next_pred", None)
                if pred is None:
                    raise KeyError(f"Model output dict missing 'next_pred'. Keys: {list(out.keys())}")
            else:
                pred = out

            pred = pred.detach().float().cpu().numpy().reshape(-1)
            y_np = y.detach().cpu().numpy().reshape(-1)

            # persistence baseline, last timestep power_norm is channel 0
            persist = X[:, -1, 0].detach().float().cpu().numpy().reshape(-1)

            y_all.append(y_np)
            yhat_all.append(pred)
            yhat_persist_all.append(persist)
            gti_raw_all.append(gti_raw.detach().cpu().numpy().reshape(-1))

    y_all = np.concatenate(y_all)
    yhat_all = np.concatenate(yhat_all)
    yhat_persist_all = np.concatenate(yhat_persist_all)
    gti_raw_all = np.concatenate(gti_raw_all)

    # all windows
    rmse_model_all = rmse(yhat_all, y_all)
    rmse_persist_all = rmse(yhat_persist_all, y_all)

    # active-only
    active = (gti_raw_all > day_thr_raw) & (y_all > power_thr)
    if active.sum() > 0:
        rmse_model_act = rmse(yhat_all[active], y_all[active])
        rmse_persist_act = rmse(yhat_persist_all[active], y_all[active])
    else:
        rmse_model_act = np.nan
        rmse_persist_act = np.nan

    return {
        "n_windows": int(len(y_all)),
        "rmse_model_all": rmse_model_all,
        "rmse_persist_all": rmse_persist_all,
        "n_active": int(active.sum()),
        "rmse_model_active": float(rmse_model_act),
        "rmse_persist_active": float(rmse_persist_act),
    }

# Load data + model
val_df = pd.read_parquet(VAL_PQ)
model = load_encoder(ENCODER_PT)

out = eval_on_val(val_df, model, day_thr_raw=10.0, power_thr=0.02, window_size=96, batch_size=512, num_workers=2)
out


[INFO] GlobalLSTMEncoder initialized:
  Input size: 20 (15 original + 5 plant IDs)
  Hidden size: 64
  Num layers: 2
  Dropout: 0.1


/tmp/ipykernel_503447/2932212832.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(weights_path, map_location="cpu")


{'n_windows': 36952,
 'rmse_model_all': 0.021465856581926346,
 'rmse_persist_all': 0.02316436916589737,
 'n_active': 8646,
 'rmse_model_active': 0.04274723678827286,
 'rmse_persist_active': 0.04634156823158264}

# Cell 5 (pretty print, quick pass/fail)

In [8]:
print("Windows:", out["n_windows"])
print(f"All windows:   model {out['rmse_model_all']:.4f} vs persist {out['rmse_persist_all']:.4f}",
      "-> OK" if out["rmse_model_all"] <= out["rmse_persist_all"] else "-> BAD")

print("Active windows:", out["n_active"])
if np.isfinite(out["rmse_model_active"]):
    print(f"Active only:   model {out['rmse_model_active']:.4f} vs persist {out['rmse_persist_active']:.4f}",
          "-> OK" if out["rmse_model_active"] <= out["rmse_persist_active"] else "-> BAD")
else:
    print("Active only: no active windows under current thresholds.")


Windows: 36952
All windows:   model 0.0215 vs persist 0.0232 -> OK
Active windows: 8646
Active only:   model 0.0427 vs persist 0.0463 -> OK
